## Notes from Calvin:

### Since we are still working on cleaning our datasets, I am testing this script using dummy data provided by Claude. Annotations are made in each code block to explain (in human language) what the code is doing.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler as Sampler
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pandas as pd
import numpy as np
import time

backend = FakeMarrakesh()

In [4]:
df = pd.read_csv('../data/processed/wildfire_cleaned_TEST.csv')
df.head()

,zip,Year,avg_tmax_c,avg_tmin_c,tot_prcp_mm,wildfire_occurred,num_fires,total_acres,max_acres,avg_acres,most_common_cause
0,90001,2018,22.319292,14.603904,198.1,0,0.0,0.0,0.0,0.0,0
1,90001,2019,21.559567,13.958907,475.7,0,0.0,0.0,0.0,0.0,0
2,90001,2020,22.298544,14.032893,229.6,0,0.0,0.0,0.0,0.0,0
3,90001,2021,20.878253,13.234925,307.3,0,0.0,0.0,0.0,0.0,0
4,90002,2018,22.319292,14.603904,198.1,0,0.0,0.0,0.0,0.0,0


In [5]:
# In this specific data provided by Claude, it reduced the dataset to 11 columns and recorded a 9:1 ratio imbalance (so a lot more no than yes)

df = df[df["Year"].between(2018, 2022)].reset_index(drop=True)

# Checking for any bad rows that managed to slip past cleaning
assert "wildfire_occurred" in df.columns, "Missing target column!"
assert df["wildfire_occurred"].isin([0, 1]).all(), "Target column has non-binary values!"
assert df.duplicated(subset=["Year", "zip"]).sum() == 0, "Duplicate Year+ZIP rows found!"
assert df["avg_tmax_c"].isnull().sum() == 0, "Null values found in weather features!"

print("Shape:", df.shape)
print("Years:", sorted(df["Year"].unique()))
print("Unique ZIPs:", df["zip"].nunique())
print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:")
print(df["wildfire_occurred"].value_counts())
print(f"Class imbalance ratio: {df['wildfire_occurred'].value_counts()[0] / df['wildfire_occurred'].value_counts()[1]:.1f}:1")

Shape: (10544, 11)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Unique ZIPs: 2593
Nulls:
 Series([], dtype: int64)

Target distribution:
wildfire_occurred
0    9498
1    1046
Name: count, dtype: int64
Class imbalance ratio: 9.1:1


In [6]:
df = df.drop(columns=["most_common_cause"]) # Dropping since column is too noisy (too many 0 entries that are insignificant to the model)

df["fire_size_risk"] = df["max_acres"] / (df["avg_acres"] + 1) # Severity relative to avg
df["temp_range"] = df["avg_tmax_c"] - df["avg_tmin_c"] # Daily temperature swing

print("Final features:")
print(df.drop(columns=["wildfire_occurred"]).columns.tolist())
print("\nShape:", df.shape)

Final features:
['zip', 'Year', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm', 'num_fires', 'total_acres', 'max_acres', 'avg_acres', 'fire_size_risk', 'temp_range']

Shape: (10544, 12)


In [7]:
# We might not need to keep this code block if OUR cleaned data is less extreme than Claude's (i don't trust Claude that much)

# Separate classes
fire = df[df["wildfire_occurred"] == 1]
no_fire = df[df["wildfire_occurred"] == 0]

# Undersample majority class to 3:1 ratio
no_fire_downsampled = resample(
    no_fire,
    replace=False,
    n_samples=len(fire) * 3,
    random_state=42
)

df = pd.concat([fire, no_fire_downsampled]).reset_index(drop=True)

print("After undersampling:")
print(df["wildfire_occurred"].value_counts())
print(f"New class ratio: {df['wildfire_occurred'].value_counts()[0] / df['wildfire_occurred'].value_counts()[1]:.1f}:1")

After undersampling:
wildfire_occurred
0    3138
1    1046
Name: count, dtype: int64
New class ratio: 3.0:1


In [8]:
# This portion of the code is implementing "lagging features". without it, our model is only looking at each specific zip code + year
# without any previous context. this way, our model can take into account other possibilities, like if a wildfire happened in 2018 but not 2019 in
# a specific zipcode.
df = df.sort_values(["zip", "Year"]).reset_index(drop=True)
df["prev_total_acres"] = df.groupby("zip")["total_acres"].shift(1).fillna(0)
df["prev_num_fires"]   = df.groupby("zip")["num_fires"].shift(1).fillna(0)
df["precip_change"]    = df.groupby("zip")["tot_prcp_mm"].diff().fillna(0)

train = df[df["Year"] <= 2021] # Training data goes from 2018-2021
test  = df[df["Year"] == 2022] # Test is 2022 because we're essentially mirroring what we're going to do when we're predicting 2023

FEATURES = [c for c in df.columns if c not in ["zip", "Year", "wildfire_occurred"]]
X_train, y_train = train[FEATURES].values, train["wildfire_occurred"].values
X_test,  y_test  = test[FEATURES].values,  test["wildfire_occurred"].values

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)
print("Features:", FEATURES)

# This is using a RandomForest algorithm. what it's essentially doing is determining which
# feature columns are most important in helping our model accurately predicting a yes/no.
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importances:\n", importances)

TOP_N = 8
top_features = importances.head(TOP_N).index.tolist()
print(f"\nTop {TOP_N} features selected:", top_features)
X_train = train[top_features].values
X_test  = test[top_features].values

# Fitting features to be within pi for qubits (because they need to be within pi I guess???)
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nFeature range after scaling:")
print("  Min:", X_train.min().round(3), " Max:", X_train.max().round(3))

Train shape: (4012, 12)
Test shape:  (172, 12)
Features: ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm', 'num_fires', 'total_acres', 'max_acres', 'avg_acres', 'fire_size_risk', 'temp_range', 'prev_total_acres', 'prev_num_fires', 'precip_change']

Feature importances:
 fire_size_risk      0.257709
max_acres           0.246176
total_acres         0.203175
avg_acres           0.168571
num_fires           0.091958
prev_total_acres    0.015461
prev_num_fires      0.011480
avg_tmin_c          0.004317
tot_prcp_mm         0.000504
avg_tmax_c          0.000306
temp_range          0.000282
precip_change       0.000061
dtype: float64

Top 8 features selected: ['fire_size_risk', 'max_acres', 'total_acres', 'avg_acres', 'num_fires', 'prev_total_acres', 'prev_num_fires', 'avg_tmin_c']

Feature range after scaling:
  Min: -3.142  Max: 3.142


In [9]:
# This is the actual quantum portion of our code. we're using a ZZFeatureMap, which basically encodes our feature columns into
# their quantum states (converting their values into quantum bits).

NUM_QUBITS = 8

feature_map = ZZFeatureMap(
    feature_dimension = 8, # How many features
    reps = 2, # Repititions
    entanglement = "linear" # How qubits connect/map to each other
)

print(f"Number of qubits : {feature_map.num_qubits}")
print(f"Circuit depth    : {feature_map.decompose().depth()}")
print(f"Number of params : {feature_map.num_parameters}")
print()
feature_map.decompose().draw("text")

# Ok this circuit looks very confusing, but it's just showing the encoding process of our features.

Number of qubits : 8
Circuit depth    : 31
Number of params : 8



C:\Users\cwang\AppData\Local\Temp\ipykernel_26076\3871309559.py:6: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  feature_map = ZZFeatureMap(


┌───┐┌───────────┐                                        ┌───┐»
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■──┤ H ├»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐└───┘»
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──■──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_6: ┤ H ├┤ P(2*x[6]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_7: ┤ H ├┤ P(2*x[7]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«             ┌───────────┐                                                 »
«q_0: ────────┤ P(2*x[0]) ├─────────────────────────────────────────────────»
«             └───────────┘              ┌───┐        ┌───────────┐         »
«q_1: ────────────────────────────────■──┤ H ├────────┤ P(2*x[1]) ├─────────»
«     ┌────────────────────────────┐┌─┴─┐└───┘        └───────────┘         »
«q_2: ┤ P(2*(π - x[1])*(π - x[2])) ├┤ X ├──■────────────────────────────────»
«     └────────────────────────────┘└───┘┌─┴─┐┌────────────────────────────┐»
«q_3: ───────────────────────────────────┤ X ├┤ P(2*(π - x[2])*(π - x[3])) ├»
«                                        └───┘└────────────────────────────┘»
«q_4: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_5: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_6: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_7: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«                                                                           »
«q_0: ──■──────────────────────────────────────────────■────────────────────»
«     ┌─┴─┐┌────────────────────────────┐            ┌─┴─┐                  »
«q_1: ┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├────────────┤ X ├───────────────■──»
«     └───┘└───────────┬───┬────────────┘        ┌───┴───┴───┐         ┌─┴─┐»
«q_2: ──■──────────────┤ H ├─────────────────────┤ P(2*x[2]) ├─────────┤ X ├»
«     ┌─┴─┐            └───┘                     └───────────┘         └───┘»
«q_3: ┤ X ├──────────────■───────────────────────────────────────────────■──»
«     └───┘            ┌─┴─┐             ┌────────────────────────────┐┌─┴─┐»
«q_4: ─────────────────┤ X ├─────────────┤ P(2*(π - x[3])*(π - x[4])) ├┤ X ├»
«                      └───┘             └────────────────────────────┘└───┘»
«q_5: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_6: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«q_7: ──────────────────────────────────────────────────────────────────────»
«                                                                           »
«                                                                      »
«q_0: ─────────────────────────────────────────────────────────────────»
«                

In [ ]:
# A sampler is just the backend that actually runs the quantum circuit. it's just like how we have to select a kernel
# to run Python.
sampler = Sampler()
fidelity = ComputeUncompute(sampler = sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map = feature_map,
    fidelity = fidelity
)

# SVC stands for Support Vector Classifier. It's a classic ML model which essentially looks at our data and finds the best boundary line
# or (divider) that separates the two classes that we're checking for (in our case, wildfire or no wildfire)
model = SVC(
    kernel = quantum_kernel.evaluate,
    probability = True, # By making this true, we make the model output a probability score instead of just yes/no
    class_weight = 'balanced',
    random_state = 42
)

Test set size: 172


### Calvin - Stopping here cause there are still some things I need to figure out. I'll work on actually training the model tomorrow.